# ️ Análise Exploratória de Dados (EDA) e Governança de Particionamento
## Classificação de Resíduos com Aprendizado Frugal (TrashNet)

**Contexto Acadêmico:** Trabalho Final — Redes Neurais Artificiais (UNIFOR)  
**Dataset de Referência:** TrashNet (2.527 imagens em 6 classes canônicas)  

### Objetivos Metodológicos deste Notebook
1. **Auditoria de Integridade Técnica:** Verificar integridade de leitura, canais cromáticos (RGB), ausência de corrupção de arquivos e dimensões nativas ($512 \times 384$ pixels) nas 2.527 amostras oficiais;
2. **Diagnóstico dos Desafios Visuais:** Analisar morfologias, reflexos especulares e ambiguidade entre classes concorrentes;
3. **Particionamento Estratificado Determinístico:** Segmentar os dados em 70% treino (1.768), 15% validação (379) e 15% teste (380) sob semente determinística `seed=42`, vedando qualquer vazamento de dados (*data leakage*);
4. **Geração das Figuras Obrigatórias (Seção 4 da Proposta):**
   - **Figura 1:** Distribuição de classes por subconjunto (gráfico de barras comparativo);
   - **Figura 2:** Painel amostral com ao menos duas imagens representativas de cada uma das seis classes;
   - **Figura 3:** Uma mesma amostra avaliada nas três resoluções espaciais ($128 \times 96$, $256 \times 192$ e $512 \times 384$ pixels);
5. **Cálculo dos Pesos de Balanceamento:** Derivar pesos inversamente proporcionais estritamente sobre a partição de treino para compensar o desequilíbrio severo da classe minoritária (*trash*).

## 1. Configuração do Repositório e Ambiente de Execução
Bootstrap autônomo compatível com Google Colab e ambientes locais. Configura diretórios, obtém dados e inicializa bibliotecas.

In [ ]:
# @title 1.1 Configuração do Repositório e Importação de Bibliotecas
import json
import os
import random
import subprocess
import sys
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

REPO_URL = "https://github.com/patryckalves/trashnet-frugal-learning.git"  # @param {type:"string"}
PROJECT_DIR = "trashnet-frugal-learning"

if not os.path.exists("src"):
    if os.path.exists(PROJECT_DIR):
        os.chdir(PROJECT_DIR)
    elif os.path.exists(f"../{PROJECT_DIR}"):
        os.chdir(f"../{PROJECT_DIR}")
    elif REPO_URL and "patryckalves" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL], check=True)
        if os.path.exists(PROJECT_DIR):
            os.chdir(PROJECT_DIR)

if "." not in sys.path:
    sys.path.insert(0, ".")

DATA_DIR = "data/raw/dataset-resized"
if not os.path.exists(DATA_DIR):
    zip_path = "dataset-resized.zip"
    if not os.path.exists(zip_path):
        url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
        subprocess.run(["curl", "-L", "-o", zip_path, url], check=True)
    subprocess.run(["unzip", "-q", zip_path, "-d", "data/raw/"], check=True)

from src.dataset import (
    CLASS_NAMES,
    CLASS_TO_IDX,
    calculate_balanced_class_weights,
    create_stratified_splits,
    export_split_metadata,
    scan_dataset,
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
print("Ambiente inicializado e dependencias carregadas com sucesso.")


In [ ]:
# @title 1.2 Controle de Aleatoriedade e Governança Metodológica
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

FIGURES_DIR = "results/figures"
SPLITS_DIR = "data/splits"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

print(f"Diretorio de dados: {DATA_DIR}")
print(f"Diretorio de figuras: {FIGURES_DIR}")
print(f"Diretorio de particoes: {SPLITS_DIR}")
print(f"Semente deterministica global: {SEED}")
print(f"Ordem canonica das classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")


## 2. Auditoria Técnica e Integridade Estrutural dos Arquivos
Inspeção minuciosa de cada arquivo contido nas seis pastas canônicas para validar integridade de decodificação, dimensões nominais e canais de cor.

In [ ]:
# @title 2.1 Varredura de Integridade e Extração de Metadados
def audit_dataset_integrity(directory_path: str) -> pd.DataFrame:
    """Realiza auditoria completa da integridade fisica das imagens do dataset.

    Args:
        directory_path: Caminho raiz para o diretorio do dataset.

    Returns:
        DataFrame com metadados estruturados de cada imagem avaliada.
    """
    df_meta = scan_dataset(directory_path)
    return df_meta


df_raw = audit_dataset_integrity(DATA_DIR)
total_samples = len(df_raw)
corrupted_count = int((~df_raw["is_valid"]).sum())
modes_found = sorted(df_raw["mode"].dropna().unique().tolist())
resolutions_found = df_raw[["width", "height"]].drop_duplicates().values.tolist()

print(f"Total de amostras catalogadas: {total_samples}")
print(f"Arquivos com falha de leitura: {corrupted_count}")
print(f"Formatos cromáticos detectados: {modes_found}")
print(f"Resoluções espaciais (largura x altura): {resolutions_found}")

summary_table = df_raw.groupby("class_name").agg(
    total=("filename", "count"),
    largura=("width", "first"),
    altura=("height", "first"),
    canais=("mode", "first"),
).loc[CLASS_NAMES]

summary_table["porcentagem"] = (summary_table["total"] / total_samples * 100).round(2)
summary_table


## 3. Diagnóstico Qualitativo dos Desafios Visuais e Desbalanceamento

A análise estatística dos metadados revela desafios centrais para o treinamento supervisionado:

### 3.1 Desequilíbrio Severo da Cauda de Distribuição
A classe `trash` conta com apenas **137 exemplos** (5,42% do volume total), enquanto `paper` concentra **594 amostras** (23,51%) e `glass` reúne **501 amostras** (19,83%). Esse desequilíbrio de magnitude superior a 4:1 impõe um risco metodológico direto: redes neurais treinadas via entropia cruzada não ponderada convergem para mínimos locais que negligenciam a classe minoritária, elevando a acurácia global à custa de taxas inaceitáveis de falso negativo em resíduos perigosos ou residuais.

### 3.2 Desafios Ópticos e Semânticos Intra e Inter-Classes
- **Ambiguidade entre Plástico e Vidro:** Recipientes transparentes de tereftalato de polietileno (PET) e garrafas de vidro mineral compartilham índices de refração aproximados, geometria cilíndrica e transparência de fundo sob luz direta;
- **Sobreposição Textural Papel vs. Papelão:** Caixas de papelão amassadas e sacolas de papel Kraft possuem idêntica pigmentação parda e fibras rugosas, demandando discernimento de rigidez e bordas estruturadas;
- **Reflexos Especulares em Metal:** Latas de alumínio sofrem reflexões anisotrópicas pontuais dependentes do ângulo de incidência do iluminante, gerando saturação localizada no sensor;
- **Heterogeneidade Extrema da Classe Trash:** Diferente das outras cinco categorias unificadas por substrato material homogêneo, a classe *trash* compreende resíduos não recicláveis diversificados (bitucas de cigarro, embalagens laminadas multicamadas, pilhas, canetas), inviabilizando a formação de um protótipo visual canônico.

## 4. Particionamento Estratificado e Prevenção de Vazamento (Data Leakage)
Em conformidade estrita com as diretrizes da Seção 2 da proposta:
- Divisão com frações canônicas: **70% treinamento**, **15% validação** e **15% teste**;
- Estratificação exata por classe garantida por semente determinística `seed=42`;
- Registro imutável em arquivos CSV (`train.csv`, `val.csv`, `test.csv`) para replicação idêntica em todos os modelos (MLP, CNN e YOLO).

In [ ]:
# @title 4.1 Execução do Particionamento Estratificado (70 / 15 / 15)
def execute_stratified_split(
    df_data: pd.DataFrame,
    seed: int = 42,
    output_dir: str = "data/splits",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide o conjunto de dados de maneira estratificada e persiste os arquivos.

    Args:
        df_data: DataFrame com todos os metadados brutos.
        seed: Semente deterministica para particionamento.
        output_dir: Diretorio para exportacao dos CSVs e metadados.

    Returns:
        Tupla contendo os DataFrames de treino, validacao e teste.
    """
    tr_df, va_df, te_df = create_stratified_splits(
        df=df_data,
        train_size=0.70,
        val_size=0.15,
        test_size=0.15,
        random_state=seed,
    )
    class_weights = calculate_balanced_class_weights(tr_df)
    export_split_metadata(tr_df, va_df, te_df, class_weights, output_dir)
    return tr_df, va_df, te_df


train_df, val_df, test_df = execute_stratified_split(df_raw, seed=SEED, output_dir=SPLITS_DIR)

print(f"Divisao concluida: Treino={len(train_df)} | Validacao={len(val_df)} | Teste={len(test_df)}")

split_proportions = pd.DataFrame({
    "Treino": train_df["class_name"].value_counts()[CLASS_NAMES],
    "Treino (%)": (train_df["class_name"].value_counts()[CLASS_NAMES] / len(train_df) * 100).round(2),
    "Validacao": val_df["class_name"].value_counts()[CLASS_NAMES],
    "Validacao (%)": (val_df["class_name"].value_counts()[CLASS_NAMES] / len(val_df) * 100).round(2),
    "Teste": test_df["class_name"].value_counts()[CLASS_NAMES],
    "Teste (%)": (test_df["class_name"].value_counts()[CLASS_NAMES] / len(test_df) * 100).round(2),
})
split_proportions


## 5. Figura Obrigatória 1: Distribuição de Classes por Subconjunto
Exigência da Seção 4 da proposta acadêmica: *"Barras com a quantidade de imagens em cada classe para treinamento, validação e teste"*.

In [ ]:
# @title 5.1 Geração e Exportação do Gráfico de Distribuição
def plot_split_distribution_chart(
    tr_df: pd.DataFrame,
    va_df: pd.DataFrame,
    te_df: pd.DataFrame,
    classes: list[str],
    output_path: str,
) -> plt.Figure:
    """Renderiza e salva o grafico de barras da distribuicao das particoes.

    Args:
        tr_df: DataFrame contendo as amostras de treino.
        va_df: DataFrame contendo as amostras de validacao.
        te_df: DataFrame contendo as amostras de teste.
        classes: Lista ordenada dos nomes canonicos das classes.
        output_path: Caminho no disco para gravacao da imagem.

    Returns:
        Objeto plt.Figure com o grafico gerado.
    """
    tr_counts = [int((tr_df["class_name"] == c).sum()) for c in classes]
    va_counts = [int((va_df["class_name"] == c).sum()) for c in classes]
    te_counts = [int((te_df["class_name"] == c).sum()) for c in classes]

    x_indices = np.arange(len(classes))
    bar_width = 0.26

    fig, ax = plt.subplots(figsize=(11, 6), dpi=150)

    rects_tr = ax.bar(x_indices - bar_width, tr_counts, bar_width, label=f"Treinamento (N={len(tr_df)})", color="#1f77b4")
    rects_va = ax.bar(x_indices, va_counts, bar_width, label=f"Validação (N={len(val_df)})", color="#ff7f0e")
    rects_te = ax.bar(x_indices + bar_width, te_counts, bar_width, label=f"Teste (N={len(test_df)})", color="#2ca02c")

    def annotate_bars(bars):
        for bar in bars:
            h = bar.get_height()
            ax.annotate(
                f"{h}",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="bold",
            )

    annotate_bars(rects_tr)
    annotate_bars(rects_va)
    annotate_bars(rects_te)

    ax.set_ylabel("Quantidade de Imagens", fontweight="bold")
    ax.set_title("Distribuição Estratificada das Classes nos Subconjuntos de Estudo", fontweight="bold", pad=14)
    ax.set_xticks(x_indices)
    ax.set_xticklabels([c.capitalize() for c in classes], fontweight="bold")
    ax.legend(frameon=True, loc="upper right")
    ax.set_ylim(0, max(tr_counts) * 1.15)
    plt.tight_layout()

    plt.savefig(output_path, bbox_inches="tight")
    return fig


fig_dist_path = os.path.join(FIGURES_DIR, "distribuicao_classes_split.png")
fig_dist = plot_split_distribution_chart(train_df, val_df, test_df, CLASS_NAMES, fig_dist_path)
print(f"Figura 1 salva com sucesso em: {fig_dist_path}")
plt.show()


## 6. Figura Obrigatória 2: Painel Amostral das Seis Categorias
Exigência da Seção 4 da proposta: *"Ao menos duas imagens de cada classe, mostrando variações visuais"*.

In [ ]:
# @title 6.1 Geração e Exportação do Painel Amostral de Classes
def plot_class_sample_panel(
    df_samples: pd.DataFrame,
    classes: list[str],
    seed: int,
    output_path: str,
) -> plt.Figure:
    """Renderiza e salva painel com duas amostras representativas por classe.

    Args:
        df_samples: DataFrame contendo caminhos das imagens e rotulos.
        classes: Lista ordenada dos nomes das classes.
        seed: Semente deterministica para selecao de amostras.
        output_path: Caminho no disco para gravacao da imagem.

    Returns:
        Objeto plt.Figure com a matriz de imagens renderizada.
    """
    fig, axes = plt.subplots(len(classes), 2, figsize=(8, 16), dpi=150)

    for row_idx, cls_name in enumerate(classes):
        selected = df_samples[df_samples["class_name"] == cls_name].sample(2, random_state=seed)
        for col_idx, (_, row) in enumerate(selected.iterrows()):
            ax = axes[row_idx, col_idx]
            img = Image.open(row["filepath"])
            ax.imshow(img)
            ax.set_title(f"Classe: {cls_name} ({row['filename']})", fontsize=10, fontweight="bold" if col_idx == 0 else "normal")
            ax.axis("off")

    plt.suptitle("Painel Amostral das Seis Classes do Dataset TrashNet", fontsize=13, fontweight="bold", y=0.99)
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight")
    return fig


fig_samples_path = os.path.join(FIGURES_DIR, "painel_exemplos_classes.png")
fig_samples = plot_class_sample_panel(train_df, CLASS_NAMES, SEED, fig_samples_path)
print(f"Figura 2 salva com sucesso em: {fig_samples_path}")
plt.show()


## 7. Figura Obrigatória 3: Avaliação Multiescala de Resolução Espacial
Exigência da Seção 4 da proposta: *"Uma mesma imagem nas três resoluções (128 × 96, 256 × 192 e 512 × 384 pixels)"*, documentando a perda progressiva de detalhes finos e o volume de tensores.

In [ ]:
# @title 7.1 Geração e Exportação da Comparação de Resoluções
def plot_resolution_scaling_comparison(
    image_filepath: str,
    resolutions: list[tuple[int, int, str]],
    output_path: str,
) -> plt.Figure:
    """Compara uma mesma amostra renderizada sob diferentes dimensoes espaciais.

    Args:
        image_filepath: Caminho do arquivo original de imagem de referencia.
        resolutions: Lista de tuplas (largura, altura, descricao).
        output_path: Caminho no disco para gravacao da imagem.

    Returns:
        Objeto plt.Figure com a comparacao multiescala.
    """
    base_img = Image.open(image_filepath)
    fig, axes = plt.subplots(1, len(resolutions), figsize=(15, 5), dpi=150)

    for idx, (width, height, label) in enumerate(resolutions):
        resized_view = base_img.resize((width, height), Image.Resampling.BILINEAR)
        ax = axes[idx]
        ax.imshow(resized_view)
        pixel_count = width * height
        ax.set_title(f"{label}\n{width} × {height} ({pixel_count:,} pixels)", fontsize=11, fontweight="bold")
        ax.axis("off")

    img_basename = os.path.basename(image_filepath)
    plt.suptitle(f"Comparativo Multiescala de Resolução Espacial ({img_basename})", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches="tight")
    return fig


reference_row = train_df[train_df["class_name"] == "glass"].iloc[0]
target_resolutions = [
    (128, 96, "Baixa Resolução (128 × 96)"),
    (256, 192, "Média Resolução (256 × 192)"),
    (512, 384, "Alta Resolução / Original (512 × 384)"),
]

fig_res_path = os.path.join(FIGURES_DIR, "comparativo_resolucoes.png")
fig_res = plot_resolution_scaling_comparison(reference_row["filepath"], target_resolutions, fig_res_path)
print(f"Figura 3 salva com sucesso em: {fig_res_path}")
plt.show()


## 8. Formulação e Cálculo dos Pesos de Classe para a Função de Perda
Conforme exigido na Seção 3 e 4 da proposta:
Para os experimentos com ponderação (E2 e E4), calculam-se pesos inversamente proporcionais à frequência das classes, **baseando-se estritamente sobre as instâncias de treinamento** para evitar contaminação metodológica:

$$\text{peso}_c = \frac{N_{\text{treino}}}{C \times N_{c,\text{treino}}}$$

Onde $N_{\text{treino}} = 1.768$ amostras, $C = 6$ classes, e $N_{c,\text{treino}}$ corresponde à contagem da classe $c$ na partição de treino.

In [ ]:
# @title 8.1 Cálculo dos Pesos Balanceados e Exportação de Parâmetros
def derive_balanced_weights_summary(
    tr_df: pd.DataFrame,
    classes: list[str],
) -> pd.DataFrame:
    """Calcula pesos balanceados no treino e estrutura tabela analitica.

    Args:
        tr_df: DataFrame com as amostras de treinamento.
        classes: Lista ordenada dos nomes das classes.

    Returns:
        DataFrame com colunas de contagem, fracao percentual e peso derivado.
    """
    weights_dict = calculate_balanced_class_weights(tr_df)
    n_train = len(tr_df)

    rows = []
    for cls_name in classes:
        count = int((tr_df["class_name"] == cls_name).sum())
        fraction = round((count / n_train) * 100, 2)
        weight = weights_dict[cls_name]
        rows.append({
            "Classe": cls_name,
            "Amostras Treino": count,
            "Fração (%)": fraction,
            "Peso Atribuído (w_c)": weight,
        })

    weights_df = pd.DataFrame(rows).set_index("Classe")
    return weights_df


df_class_weights = derive_balanced_weights_summary(train_df, CLASS_NAMES)
print("Pesos de Classe Inversamente Proporcionais Derivados do Treino:")
print(df_class_weights.to_string())

weights_json_file = os.path.join(SPLITS_DIR, "class_weights.json")
with open(weights_json_file, "r", encoding="utf-8") as f:
    saved_weights = json.load(f)
print(f"Pesos validados no manifesto JSON: {weights_json_file}")


## 9. Síntese Executiva dos Resultados da Análise Exploratória

A execução sistemática deste protocolo de análise exploratória consolida quatro garantias científicas essenciais:

1. **Integridade Estrutural Total:** Todas as 2.527 imagens do arquivo oficial `dataset-resized.zip` encontram-se íntegras, em modo RGB tridimensional nativo e com dimensões de $512 \times 384$ pixels;
2. **Blindagem Metodológica Anti-Vazamento:** A partição estratificada 70/15/15 (1.768 treino, 379 validação, 380 teste) gerada sob `seed=42` foi registrada em `data/splits/`. Todos os experimentos subsequentes consumirão rigorosamente as mesmas partições;
3. **Ponderação de Gradiente Calibrada:** A classe minoritária `trash` recebeu fator de ponderação corretivo aproximado de **3,07**, enquanto categorias majoritárias como `paper` receberam **0,71**, equalizando a influência dos gradientes nos experimentos E2 e E4;
4. **Artefatos e Figuras Persistidos:** As Figuras 1, 2 e 3 foram geradas e persistidas em `results/figures/` em conformidade integral com a Seção 4 da proposta da atividade.